In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import os
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# ==========================================
# 0. Environment and parameter configuration
# ==========================================
# Scaling factor: map the ~1e-7 concentrations to ~[0, 10] to make the symbolic-regression search easier
SCALE = 1e7
OUTPUT_DIR = "Stage1_Exploration"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# PySR PySR hyperparameters
# Hyperparameters controlling breadth and depth of the symbolic-regression search
PYSR_PARAMS = {
    "niterations": 150,           # Iteration count - higher = longer search, usually better results
    "populations": 30,            # Number of populations searched in parallel
    "maxsize": 25,                # Max formula length - prevents overly complex / overfitting expressions
    "binary_operators": ["+", "*", "-", "/"], # Allowed binary operators
    # Allowed unary operators (exp, log, sqrt, ...) typical of physical models
    "unary_operators": ["exp", "log", "sqrt", "square", "inv(x)=1/x"],
    "extra_sympy_mappings": {'inv': lambda x: 1/x}, # Custom operator mapping for SymPy
    "model_selection": "best",    # Model-selection strategy: pick the best-scoring expression
    "temp_equation_file": True,   # Save intermediate equation files so nothing is lost on interruption
    "delete_tempfiles": False,    # Keep temp files for debugging
    "verbosity": 1                # Enable logging to watch search progress
}

# ==========================================
# 1. Data loading and preprocessing
# ==========================================
print("Loading data and performing a 7:3 case-wise split...")
# Read the cleaned dataset
df = pd.read_csv('data/train_dataset_ready.csv')

# Concentration pre-scaling
# Bring the physical scales into a numerically friendly range
df['C_in'] = df['C_in'] * SCALE
df['C_out'] = df['C_out'] * SCALE

# Group split by case
# Crucial: split by case, not by individual row
# This avoids"data leakage" (Data Leakage) - neighbouring points within the same case are strongly correlated
unique_cases = df['Case'].unique()
train_cases, test_cases = train_test_split(unique_cases, test_size=0.3, random_state=42)

# Filter rows by Case ID
train_df = df[df['Case'].isin(train_cases)].copy()
test_df = df[df['Case'].isin(test_cases)].copy()

print(f"Train cases: {len(train_cases)}, test cases: {len(test_cases)}")

# Build feature matrix X and target vector y
feature_names = ['V_in', 'C_in', 'Area', 'Distance']
X_train = train_df[feature_names].values
y_train = train_df['C_out'].values
X_test = test_df[feature_names].values
y_test = test_df['C_out'].values

# Subsampling
# To balance PySR speed and search depth, draw 5000 random training samples
# (symbolic regression is not that sensitive to sample count; 5000 usually captures the physics and more just slows evolution)
np.random.seed(42)
sub_idx = np.random.choice(len(X_train), 5000, replace=False)
X_train_sub = X_train[sub_idx]
y_train_sub = y_train[sub_idx]

# ==========================================
# 2. Run PySR (symbolic regression)
# ==========================================
print("\n>>> Starting Stage 1: pure data-driven exploration...")

# Initialise the regressor
model = PySRRegressor(**PYSR_PARAMS)

# Begin fitting
# variable_names So the printed formula uses physical variable names instead of x0, x1, ...
model.fit(X_train_sub, y_train_sub, variable_names=feature_names)

# ==========================================
# 3. Evaluation and saving
# ==========================================
# 1. Save every candidate formula to CSV for later analysis
equations_path = os.path.join(OUTPUT_DIR, "stage1_all_equations.csv")
model.equations_.to_csv(equations_path)

# 2. Evaluate generalisation on the full test set
# Note: test cases are fully unseen - this is a realistic measure of generalisation
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print("\n" + "="*40)
print(f"Stage 1 exploration complete!")
print(f"Full test-set R^2 (unseen cases): {r2:.5f}")
print(f"All candidate formulas saved to: {equations_path}")
print("="*40)

# Print the PySR best pick (optimum on the Pareto front)
print("\nRecommended best formula (SymPy form):")
print(model.sympy())